# Fine-tuning a Transformer for Arabic → German Translation (Local Jupyter version)

Same structure as before, adapted to run in a **local Jupyter Notebook / JupyterLab** install instead of Colab or Kaggle:

- **Dataset:** [`Helsinki-NLP/multiun`](https://huggingface.co/datasets/Helsinki-NLP/multiun), config `ar-de` (~165k parallel sentence pairs from UN documents, streamed directly from the Hugging Face Hub).
- **Base model:** [`facebook/mbart-large-50-many-to-many-mmt`](https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt), a multilingual Seq2Seq transformer that already supports Arabic and German, which we fine-tune specifically on the ar→de direction.

> Note: there is no ready-made Helsinki-NLP `opus-mt-ar-de` checkpoint on the Hub (only the reverse `de-ar` direction exists), so mBART-50 is used as the pretrained starting point instead.

### Before running locally
1. **Python 3.9+** and a working `pip`. Ideally use a virtual environment:
   ```bash
   python -m venv .venv
   source .venv/bin/activate        # Windows: .venv\Scripts\activate
   pip install jupyter
   jupyter notebook
   ```
2. **GPU strongly recommended.** mBART-large-50 has ~610M parameters — fine-tuning on CPU only is possible but very slow (could take days for the full dataset/epochs below). If you only have a CPU:
   - Install the CPU build of PyTorch (see cell 1), and
   - Consider shrinking the run for a first test: reduce `num_train_epochs`, or subsample `raw_dataset` (e.g. `raw_dataset["train"].select(range(2000))`) before tokenizing.
3. If you do have an NVIDIA GPU, install the matching CUDA build of PyTorch from https://pytorch.org/get-started/locally/ *before* running cell 1, so `pip` doesn't fall back to the CPU-only wheel.
4. All outputs (checkpoints + final model) are saved to `./ar-de-transformer-finetuned` in the same folder as this notebook.

In [1]:
# Core ML libraries (skip/edit this line if you already installed a GPU-specific
# build of torch from https://pytorch.org/get-started/locally/)
%pip install -q torch transformers datasets accelerate

# Supporting libraries used in this notebook
%pip install -q evaluate sentencepiece sacrebleu ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import the random module
import random

# Import NumPy for numerical operations
import numpy as np

# Import the evaluation library to calculate model performance
import evaluate

# Import function to load datasets
from datasets import load_dataset

# Import Hugging Face Transformers classes
from transformers import (
    AutoTokenizer,            # Converts text into tokens the model can understand
    AutoModelForSeq2SeqLM,    # Loads a sequence-to-sequence model (e.g., mBART)
    DataCollatorForSeq2Seq,   # Prepares and batches data for training
    Seq2SeqTrainingArguments, # Defines training settings (epochs, batch size, etc.)
    Seq2SeqTrainer,           # Handles the training and evaluation process
)

In [3]:
import torch

# Quick sanity check: confirm whether a GPU is visible to PyTorch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- training will run on CPU and will be slow for this model size.")


CUDA available: True
GPU: Tesla T4


In [4]:
# Name of the pre-trained multilingual translation model (supports many-to-many translation, including ar->de)
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

# Hugging Face dataset name and language-pair configuration
DATASET_NAME = "Helsinki-NLP/multiun"
DATASET_CONFIG = "ar-de"

# Source (input) and target (output) language keys inside the dataset's "translation" dict
SOURCE_LANG = "ar"
TARGET_LANG = "de"

# mBART-50 language codes (needed for its tokenizer / generation)
SOURCE_LANG_CODE = "ar_AR"
TARGET_LANG_CODE = "de_DE"

# Maximum number of tokens for each input/output sequence
MAX_LEN = 128

# Where to save checkpoints and the final model (relative to this notebook's folder)
OUTPUT_DIR = "./ar-de-transformer-finetuned"

In [5]:
# Load the tokenizer for the pre-trained translation model
# src_lang / tgt_lang tell the mBART-50 tokenizer which language codes to use
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=SOURCE_LANG_CODE,
    tgt_lang=TARGET_LANG_CODE,
)

# Load the pre-trained sequence-to-sequence translation model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Make sure generated sequences always start with the German language token
# (must be set on generation_config, not config -- transformers no longer allows
# controlling generation via model.config)
model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [6]:
# Load the Arabic-German dataset directly from the Hugging Face Hub
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)

# Split the dataset into training (90%) and testing (10%)
raw_dataset = raw_dataset["train"].train_test_split(test_size=0.1, seed=42)

README.md:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

ar-de/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 41.1MB            

ar-de/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/165090 [00:00<?, ? examples/s]

In [7]:
# Convert the input and target texts into tokens
def preprocess_function(examples):

    # Get the source (Arabic) and target (German) sentences from the nested "translation" dict
    inputs = [pair[SOURCE_LANG] for pair in examples["translation"]]
    targets = [pair[TARGET_LANG] for pair in examples["translation"]]

    # Tokenize the input sentences
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LEN,   # Maximum sequence length
        truncation=True       # Truncate long sentences
    )

    # Tokenize the target sentences
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_LEN,   # Maximum sequence length
        truncation=True       # Truncate long sentences
    )

    # Store the target token IDs as labels
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# Apply preprocessing to the entire dataset
tokenized_dataset = raw_dataset.map(
    preprocess_function,
    batched=True,                                # Process multiple examples at once
    remove_columns=raw_dataset["train"].column_names,  # Remove original text columns
)

Map:   0%|          | 0/148581 [00:00<?, ? examples/s]

Map:   0%|          | 0/16509 [00:00<?, ? examples/s]

In [8]:
# Create a data collator to prepare batches for training
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,  # Tokenizer used for padding
    model=model           # Model used for sequence-to-sequence tasks
)

In [9]:
# Load the SacreBLEU metric for evaluating translation quality
bleu_metric = evaluate.load("sacrebleu")

In [10]:
# Clean the predicted and reference texts
def postprocess_text(preds, labels):

    # Remove extra spaces from predictions like padding
    preds = [pred.strip() for pred in preds]

    # Remove extra spaces from labels and format them for BLEU
    labels = [[label.strip()] for label in labels]

    return preds, labels


# Calculate the BLEU score during evaluation
def compute_metrics(eval_preds):

    # Get model predictions and true labels
    preds, labels = eval_preds

    # If predictions are returned as a tuple, use the first element
    if isinstance(preds, tuple):
        preds = preds[0]

    # Convert predicted token IDs back to text
    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True
    )

    # Replace ignored label values (-100) with the padding token
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Convert label token IDs back to text
    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    # Clean the decoded text
    decoded_preds, decoded_labels = postprocess_text(
        decoded_preds,
        decoded_labels
    )

    # Compute the BLEU score
    result = bleu_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # Return the BLEU score
    return {"bleu": result["score"]}

In [11]:
# Define the training settings
training_args = Seq2SeqTrainingArguments(

    # Folder to save the trained model
    output_dir=OUTPUT_DIR,

    # Evaluate the model after each training epoch
    eval_strategy="steps",

    # Save the model after each training epoch
    save_strategy="steps",


    eval_steps=200,
    save_steps=200,

    # Learning rate for the optimizer
    learning_rate=2e-5,

    # Number of training samples per GPU/CPU batch (mBART-large is big, so we use a smaller batch size
    # than a small MarianMT model and make up for it with gradient accumulation)
    per_device_train_batch_size=4,

    # Number of evaluation samples per GPU/CPU batch
    per_device_eval_batch_size=4,

    # Accumulate gradients to reach an effective batch size of 16
    gradient_accumulation_steps=4,

    # Weight decay for regularization
    weight_decay=0.01,

    # Total number of training epochs
    num_train_epochs=2,

    # Generate translations during evaluation
    predict_with_generate=True,

    # Cap generation length during evaluation to match MAX_LEN
    generation_max_length=MAX_LEN,

    # Use mixed-precision (FP16) training if supported by the GPU
    fp16=torch.cuda.is_available(),

    # Log training progress every 50 steps
    logging_steps=50,

    # Load the best-performing model after training
    load_best_model_at_end=True,

    # Select the best model based on the BLEU score
    metric_for_best_model="bleu",

    # Avoid filling up local disk with too many checkpoints
    save_total_limit=2,

    # No experiment-tracking backend configured locally
    report_to="none",
)

In [12]:
# Create the Trainer object for training and evaluation
trainer = Seq2SeqTrainer(

    # The pre-trained translation model
    model=model,

    # Training configuration (epochs, batch size, learning rate, etc.)
    args=training_args,

    # Training dataset
    train_dataset=tokenized_dataset["train"],

    # Evaluation (test) dataset
    eval_dataset=tokenized_dataset["test"],

    # Prepares batches by padding sequences automatically
    data_collator=data_collator,

    # Function to calculate the BLEU score during evaluation
    compute_metrics=compute_metrics,
)

In [13]:
# Translate an Arabic sentence into German
def translate(text, max_length=MAX_LEN):

    # Make sure the tokenizer knows the input is Arabic
    tokenizer.src_lang = SOURCE_LANG_CODE

    # Convert the input text into tokens
    inputs = tokenizer(
        text,
        return_tensors="pt",  # Return PyTorch tensors
        truncation=True,      # Truncate long sentences
        max_length=max_length # Maximum sequence length
    )

    # Move the input tensors to the same device as the model (CPU/GPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate the translated sentence, forcing German as the output language
    generated = model.generate(
        **inputs,
        max_length=max_length,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)
    )

    # Convert the generated tokens back to readable text
    return tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )

In [ ]:
# Run the code only if this file is executed directly
if __name__ == "__main__":

    # Train the translation model
    trainer.train()

    # Save the trained model
    trainer.save_model(f"{OUTPUT_DIR}/final")

    # Save the tokenizer
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")


    # Evaluate the model on the test dataset
    print("\n===== Evaluation on test set =====")
    eval_results = trainer.evaluate()

    # Print the evaluation results (e.g., BLEU score)
    for key, value in eval_results.items():
        print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")


    # Display some sample translations
    print("\n===== Sample predictions =====")

    # Get the test dataset
    test_raw = raw_dataset["test"]

    # Select up to 20 random test examples
    num_samples = min(20, len(test_raw))
    sample_indices = random.sample(range(len(test_raw)), num_samples)

    # Translate and compare each sample
    for idx in sample_indices:

        # Original Arabic sentence
        arabic_sentence = test_raw[idx]["translation"][SOURCE_LANG]

        # Correct German translation
        actual_german = test_raw[idx]["translation"][TARGET_LANG]

        # Model prediction
        predicted_german = translate(arabic_sentence)

        # Print the results
        print(f"Arabic    : {arabic_sentence}")
        print(f"Actual DE : {actual_german}")
        print(f"Predicted : {predicted_german}")
        print()


    # Translate a custom sentence
    custom_sentence = "الباب لن يفتح."

    print("===== Custom sentence =====")
    print("Arabic:", custom_sentence)
    print("German:", translate(custom_sentence))

Step,Training Loss,Validation Loss,Bleu
200,4.406413,1.033279,35.770870
400,3.986187,0.939101,38.861034
600,3.851957,0.886329,40.406227
800,3.585748,0.848808,41.785207
1000,3.382281,0.823861,42.431588
1200,3.285404,0.797693,43.681787
1400,3.206635,0.781668,44.317057
1600,3.058908,0.770109,44.074628
1800,3.206802,0.753507,45.141039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]